In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([

    # TODO: Resize to 28x28
    transforms.Resize((28, 28)),

    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)

    # TODO: Convert to Tensor
    transforms.ToTensor(),

    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),

])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s,EfficientNet_V2_S_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"
# to check if there is gpu or not, if not use cpu

efficientnet = efficientnet_v2_s(pretrained=True)
efficientnet.train().to(device)

In [ ]:
# freeze the backbone
efficientnet.requires_grad_(False)


In [ ]:
# Replace the classifier head to match the number of classes in EMNIST letters (26 classes)

in_num = efficientnet.classifier[1].in_features
out_num = efficientnet.classifier[1].out_features

print(f"old number of output {out_num}")

efficientnet.classifier[1] = nn.Linear(in_num, 26)


out_num = efficientnet.classifier[1].out_features

print(f"new number of output {out_num}")



In [ ]:
efficientnet = efficientnet.to(device)

In [ ]:
from tqdm import tqdm    # Shows progress bar

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train() # make it train to train it
    running_loss = 0.0

    for images, targets in tqdm(loader):
        # 1. Move to Device
        images = images.to(device)
        targets = targets.to(device)

        targets.long()

        # 2. Zero Gradients
        optimizer.zero_grad()

        # 3. Forward Pass
        outputs = model(images)

        # 4. Compute Loss
        loss = criterion(outputs, targets)

        # 5. Backward Pass
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)


In [ ]:
def validate(model, loader, criterion, device):
    model.eval() # Set to eval
    running_loss = 0.0

    with torch.no_grad(): # Disable gradient calculation
        for images, targets in loader:
            images = images.to(device)
            targets = targets.to(device)

            outputs = model(images)
            loss = criterion(outputs, targets)

            running_loss += loss.item()

    return running_loss / len(loader)

In [ ]:
# Write your code here
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()


# arrays to store some vars
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

model = efficientnet

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)


for epoch in range(10):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{10}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


In [ ]:
# Write your code here
